In [2]:
# use cmipv2 env

In [3]:
import numpy as np
import xarray as xr
import os
import glob
import matplotlib.pyplot as plt
import pandas as pd
import json
import cftime
from cftime import DatetimeNoLeap
from scipy import stats
from scipy.interpolate import interp1d
import dask
from tqdm import tqdm
#from mpl_axes_aligner import align
from xmip.preprocessing import rename_cmip6
%matplotlib inline
import xesmf as xe

from utils import QACC_utils
from utils.config import model, min_lat, max_lat, min_year_late_cent, max_year_late_cent

In [6]:
# old, don't need these vars here

# get the spatial mean variables (monthly) 
# - but note we also need some variables with spatial resolution for kernel cals

ds_sm_ssp245 = xr.open_dataset('intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
            s='ssp245', l=str(min_lat), m=model))
ds_sm_arise = xr.open_dataset('intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
            s='ARISE', l=str(min_lat), m=model))


In [7]:
# spatial vars for kernel cals:
spatial_vars = ['ta', 'hus'] # need temp, water vapour, and albedo
spatial_vars_sf = ['tas', 'rsus', 'rsds'] # need temp, water vapour, and albedo


ds_ssp245 = QACC_utils.get_all_vars_spatial_arctic_monthly(vars=spatial_vars, model=model, scenario='ssp245',
                                                           min_lat=min_lat, max_lat=max_lat, 
                                                           min_year=min_year_late_cent, max_year=max_year_late_cent)

ds_arise = QACC_utils.get_all_vars_spatial_arctic_monthly(vars=spatial_vars, model=model, scenario='ARISE',
                                                           min_lat=min_lat, max_lat=max_lat, 
                                                           min_year=min_year_late_cent, max_year=max_year_late_cent)

ds_ssp245, ds_arise = ds_ssp245.rename({'x':'lon', 'y':'lat'}), ds_arise.rename({'x':'lon', 'y':'lat'})


# drop the stratosphere from both, using the crude approach of a cutoff at the annual and spatial arctic mean pressure
ptp = ds_sm_ssp245['ptp'].mean().values
print(ptp)
# monthly variation is small given the low vertical resolution of our data and kernels, as shown in the plot from the line below:
#ds_sm_ssp245['ptp'].plot()
ds_ssp245 = ds_ssp245.where(ds_ssp245.plev>ptp, drop=True)
ds_arise = ds_arise.where(ds_arise.plev>ptp, drop=True)


# also get the surface onces:
ds_ssp245_sf = QACC_utils.get_all_vars_spatial_arctic_monthly(vars=spatial_vars_sf, model=model, scenario='ssp245',
                                                           min_lat=min_lat, max_lat=max_lat, 
                                                           min_year=min_year_late_cent, max_year=max_year_late_cent)

ds_arise_sf = QACC_utils.get_all_vars_spatial_arctic_monthly(vars=spatial_vars_sf, model=model, scenario='ARISE',
                                                           min_lat=min_lat, max_lat=max_lat, 
                                                           min_year=min_year_late_cent, max_year=max_year_late_cent)

ds_ssp245_sf, ds_arise_sf = ds_ssp245_sf.rename({'x':'lon', 'y':'lat'}), ds_arise_sf.rename({'x':'lon', 'y':'lat'})

ds_ssp245 = xr.merge([ds_ssp245, ds_ssp245_sf], compat='override') # only need to override compat because of the ens mems var, which is not needed
ds_arise = xr.merge([ds_arise, ds_arise_sf])

## also need to add albedo, defined as ratio of reflected up to incident down SW radiation
ds_ssp245['albedo'] = ds_ssp245['rsus']/ds_ssp245['rsds']
ds_arise['albedo'] = ds_arise['rsus']/ds_arise['rsds']



  0%|          | 0/2 [00:00<?, ?it/s]

ta


 50%|█████     | 1/2 [00:01<00:01,  1.21s/it]

hus


  0%|          | 0/2 [00:00<?, ?it/s]

...

 33%|███▎      | 1/3 [00:01<00:03,  1.82s/it]

rsus


 67%|██████▋   | 2/3 [00:04<00:02,  2.10s/it]

rsds


100%|██████████| 3/3 [00:06<00:00,  2.12s/it]
/home/users/a_duffey/.conda/envs/cmipv2/lib/python3.12/site-packages/dask/core.py:133: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


In [8]:
ds_ssp245.to_netcdf('intermediate_outputs/for_kernel_decomp/{s}_{l}_{m}.nc'.format(
        s='ssp245', l=str(min_lat), m=model))
ds_arise.to_netcdf('intermediate_outputs/for_kernel_decomp/{s}_{l}_{m}.nc'.format(
        s='ARISE', l=str(min_lat), m=model))


In [ ]:
## ignore below - old version before using climkern

In [ ]:
"""
### Radiative kernels

Kernels are from Smith et al., 2020, for HadGEM3-GA7.1: https://essd.copernicus.org/articles/12/2157/2020/
Downloaded from https://zenodo.org/records/3594673 on 19th Sep 2025

See also Pendergrass' helpful Github with some Matlab code https://github.com/apendergrass/cam5-kernels/blob/master/scripts/kernel_demo.m
"""

In [ ]:
"""
kernels = xr.open_dataset('data/kernels_HadGem3/HadGEM3-GA7.1_TOA_kernel_L19.nc')
#kernels = kernels.where(kernels.plev>ptp, drop=True).load()
"""

In [ ]:
# old version using Hadgem2 kernels
"""
kernels17 = xr.open_dataset('data/kernels_HadGem3/HadGEM2_net_TOA_L17.nc') # note that kernels stop at tropopause, hence only 17 levels, but these match the first 17 levels of the UKESM data
kernels38 = xr.open_dataset('data/kernels_HadGem2/HadGEM2_net_TOA_L38.nc') # need both as the surface ones are only included in L38 version

kernels_s = kernels38[['tsurf', 'tsurf_cs', 'albedo', 'albedo_cs']]

# drop stratosphere from kernels on levels, as for the other data
kernels17 = kernels17.where(kernels17.plev>ptp, drop=True)
kernels17 = kernels17.isel(plev=slice(None, None, -1)) # also reverse the plev dim in kernels, to align with model data

kernels = xr.merge([kernels17, kernels_s])

# select only the arctic
kernels = kernels.sel(lat=slice(min_lat, max_lat))

# regrid to the model data
regridder = xe.Regridder(kernels, ds_ssp245, 'bilinear',extrap_method="nearest_s2d")
kernels = regridder(kernels)
kernels
"""

In [ ]:
"""
## Okay, lets calc some Rs

# albedo
delta_albedo_perc = 100*(ds_arise['albedo'] - ds_ssp245['albedo'])/ds_ssp245['albedo']
R_albedo = delta_albedo_perc*kernels['albedo_sw']

# 
R_albedo
"""

In [ ]:
#R_albedo.mean('month').plot()